# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates step-by-step how to load and explore the FAIR\^2 colorectal cancer survivors dataset using the `mlcroissant` library and Python tools.

### Dataset Source
This dataset and Croissant schema are available at:
[https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset loaded: {metadata.name}\n")
print("Description:")
print(metadata.description)

# Optionally, show available metadata fields
print(f"\nAvailable metadata attributes: {', '.join([f for f in dir(metadata) if not f.startswith('_')])}")

## 2. Data Overview
Review available record sets, fields, and their IDs (using the Croissant `@id` for each entity).

Let's inspect which record sets are provided by the dataset and print out their fields and columns by `@id`.

In [ ]:
# Get all recordset @ids in the dataset
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
print(f"Found {len(record_set_ids)} record sets.\n")
for record_set in metadata.record_sets:
    print(f"Record Set: {record_set['@id']}")
    print(f"  Name: {record_set['name']}")
    print(f"  Description: {record_set.get('description', '')}")
    print(f"  Fields:")
    for field in record_set.get('fields', []):
        print(f"    - Field @id: {field['@id']} | Name: {field['name']}")
        if 'columns' in field:
            print("      Columns:")
            for column in field['columns']:
                print(f"        - Column @id: {column['@id']} | Name: {column['name']}")
    print("\n")
# For further cells, we'll pick one record set for example exploration below:
if len(record_set_ids) > 0:
    default_record_set_id = record_set_ids[0]
else:
    default_record_set_id = None

## 3. Data Extraction
Load tabular data from a specific record set into a DataFrame for analysis, using the record set and field `@id`s found above.

In [ ]:
# Extract all record sets by @id into Pandas DataFrames
dataframes = dict()
for record_set_id in record_set_ids:
    print(f"Loading data from record set '@id': {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  Loaded {len(df)} records.\n")
    except Exception as e:
        print(f"  WARNING: Failed to load {record_set_id}: {e}\n")
# Pick a default record set for demonstration
if default_record_set_id:
    print(f"\nFirst 5 rows of main record set '@id': {default_record_set_id}")
    display(dataframes[default_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps on the dataset. We'll select a numeric field (column) by `@id`, filter records, normalize, and group by a categorical field (also referenced by `@id`).

_You may need to adjust the example depending on the actual record set/field IDs and field types printed above._

In [ ]:
# For the demonstration, let's pick a numeric and a grouping field from the first record set
example_df = dataframes[default_record_set_id]

# Inspect column names to pick fields by Croissant @id
print("All columns (@id):", example_df.columns.tolist())

# Try to guess some likely candidates for numeric and categorical fields by name (fallback: first numeric and first category)
numeric_field_candidates = [col for col in example_df.columns if ('age' in col.lower() or 'interval' in col.lower() or example_df[col].dtype in [int, float])]
if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]
else:
    numeric_field_id = example_df.select_dtypes(['int64', 'float64']).columns[0] if not example_df.select_dtypes(['int64', 'float64']).empty else example_df.columns[0]

group_field_candidates = [col for col in example_df.columns if 'sex' in col.lower() or 'status' in col.lower() or 'location' in col.lower() or example_df[col].dtype == object]
if group_field_candidates:
    group_field_id = group_field_candidates[0]
else:
    group_field_id = example_df.select_dtypes(['object']).columns[0] if not example_df.select_dtypes(['object']).empty else example_df.columns[0]

print(f"Numeric field chosen for analysis: {numeric_field_id}")
print(f"Group-by field chosen: {group_field_id}")

# Filter: select records where numeric_field > threshold
threshold = example_df[numeric_field_id].mean()  # Use mean as a threshold
filtered_df = example_df[example_df[numeric_field_id] > threshold]

print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize the numeric field
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nFirst 5 normalized values for {numeric_field_id}:")
display(filtered_df[[numeric_field_id, norm_col]].head())

# Group by the selected field and show means
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
    print(f"\nMean {numeric_field_id} by {group_field_id} (filtered records):")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of a numeric field and/or a relationship between the selected numeric and categorical fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(example_df[numeric_field_id].dropna(), kde=True, bins=15)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot for numeric field by group field, if group field is reasonably categorical
if group_field_id in example_df.columns and example_df[group_field_id].nunique() <= 10:
    plt.figure(figsize=(9, 5))
    sns.boxplot(data=example_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to use the `mlcroissant` library to:
- Load metadata and record sets from a FAIR-compliant Croissant dataset using only `@id` references,
- Explore available record sets, fields, and columns,
- Extract tabular records into pandas DataFrames,
- Perform basic EDA: filtering, normalization, and grouping operations,
- Visualize data distributions and relationships.

This workflow can be reused for other Croissant datasets simply by changing the data URL. For further biological, clinical, or machine learning analysis, apply domain-appropriate tools to the resulting DataFrames.

**Key lessons:**
- Always refer to record sets, fields, and columns by their Croissant `@id` for robust, schema-compliant data extraction.
- The `mlcroissant` package can flexibly handle datasets with multiple record sets, complex inheritance, and FAIR metadata.

[Learn more about Croissant metadata schema](https://mlcommons.github.io/croissant/).